# Codebook workflow
Run one step, review its results, then decide what to do next. One session folder represents one experiment. The Python implementation is in `simple_experiment.py`.

**Dataset numbers identify input files. Codebook versions identify successive codebooks.** Version numbers only go up: 1, 2, 3 across all datasets and models. There is no resetting revision counter.

Every request starts with `preview=True`. It saves the request locally and prints a short summary. Change that call to `preview=False` when ready to submit. Review cells never submit requests. For batch mode, rerun the review cell until ready. Keep optional steps unrun until you decide you need them.


In [ ]:
import os
from dotenv import load_dotenv
from simple_experiment import Experiment

load_dotenv("../.env")

experiment = Experiment(
    session_dir="emotive_rhetoric",
    model_a="openai/gpt-6-luna",
    model_b="deepseek/deepseek-v4.1-flash",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    mode="stream",  # "stream" runs now; "batch" submits work to check later.
    n_runs=25,
)


## 1. Initial coding
Code dataset 0 without a codebook. Existing runs are reused if you rerun the submission cell. Review prints alpha and shows document agreement.


In [ ]:
experiment.code(datasets=0, preview=True)


In [ ]:
experiment.review(datasets=0)


## 2. Generate the first codebook
Generate version 1 using the initial coding summaries. Review displays the definition and numbered rules. The saved path is captured without dumping raw API JSON.


In [ ]:
experiment.generate(datasets=0, version=1, preview=True)


In [ ]:
codebook_path = experiment.show_codebook(version=1)


## 3. Check the current codebook
This is the main work area. Choose a codebook version and the datasets to check here. After any revision, update `current_version` to the new number and rerun these cells.

`datasets=0` checks one dataset; `[0, 1]` checks both together. Every document ID must be unique across the selected files.


In [ ]:
current_version = 1
datasets_to_check = [0]


In [ ]:
experiment.code(datasets=datasets_to_check, version=current_version, preview=True)


In [ ]:
experiment.review(datasets=datasets_to_check, version=current_version)


### Decision: keep it or revise it
If satisfied, run the acceptance cell. Acceptance records your decision; it does not create another codebook version. Replace the note with your reason.


In [ ]:
# Optional: run only when you choose to keep this version.
# experiment.accept(version=current_version, note='Agreement is sufficient to move on.')


If you need a revision, set the next unused version number explicitly. **Rerunning the same version resumes or reviews that task; it never increments the version automatically.** Choose a number higher than every version already used (see `experiment.status()`).

After reviewing the new codebook, update `current_version` above, then rerun the coding and review cells. There is no need to copy paths or load JSON manually.


In [ ]:
new_version = 2


In [ ]:
experiment.revise(
    datasets=datasets_to_check,
    source_version=current_version,
    version=new_version,
    preview=True,
)


In [ ]:
revised_codebook_path = experiment.show_codebook(version=new_version)


## 4. Test a new dataset
Use the current accepted version on a new dataset. Change `next_dataset` to the input file number you want to inspect.


In [ ]:
next_dataset = 1


In [ ]:
experiment.code(datasets=next_dataset, version=current_version, preview=True)


In [ ]:
experiment.review(datasets=next_dataset, version=current_version)


If agreement is low, make an additions-only revision using the new dataset. Choose the next unused version number. After reviewing it, update `current_version` in section 3 and set `datasets_to_check = [0, 1]` (or all datasets seen so far) to check the complete collection.


In [ ]:
# Optional: uncomment when a new dataset requires revision.
# new_version = 3  # Use the next unused number.
# experiment.revise(
#     datasets=next_dataset, source_version=current_version,
#     version=new_version, additions_only=True, preview=True,
# )


In [ ]:
# revised_codebook_path = experiment.show_codebook(version=new_version)


## 5. Compare models
Choose the same dataset collection and codebook for both models. Model A's existing coding is reused when available. Review each model, then compare their distributions and paired-run alpha.

The comparison prints mean absolute difference in proportions, Pearson correlation, and average nominal Krippendorff alpha for matching A/B run numbers. It shows a scatterplot with a diagonal and slight jitter. Analysis outputs are displayed without saving tables or figures.


In [ ]:
comparison_datasets = [0, 1]


In [ ]:
experiment.code(datasets=comparison_datasets, version=current_version, model="A", preview=True)


In [ ]:
experiment.code(datasets=comparison_datasets, version=current_version, model="B", preview=True)


In [ ]:
experiment.review(datasets=comparison_datasets, version=current_version, model="A")


In [ ]:
experiment.review(datasets=comparison_datasets, version=current_version, model="B")


In [ ]:
experiment.compare(datasets=comparison_datasets, version=current_version)


If cross-model disagreement requires revision, use both models' coding summaries. Review the revised codebook, update `current_version`, and rerun both models on all datasets seen so far. Continue using the same increasing version sequence.


In [ ]:
# Optional: uncomment if cross-model disagreement requires revision.
# new_version = 4  # Use the next unused number.
# experiment.revise(
#     datasets=comparison_datasets, source_version=current_version,
#     version=new_version, compare_models=True, preview=True,
# )


In [ ]:
# revised_codebook_path = experiment.show_codebook(version=new_version)


## Resume or inspect progress
After a kernel restart, rerun setup and set `current_version` and your dataset selections. All requests and results remain on disk. This status cell reads local files without API calls.

Saved files live in `emotive_rhetoric/out/`:

```text
coding/datasets-0/no-codebook/model-A/       initial coding
coding/datasets-0/version-001/model-A/      coding using version 1
coding/datasets-0-1/version-002/model-B/    cumulative coding by B
codebooks/version-001/codebook.json         full codebook
codebooks/version-002/documentation.json    revision notes
orders/datasets-0.json                      reusable plot order
```

Request/response files and input snapshots stay inside their task folders. Each task records its inputs and rejects changed settings rather than mixing results. To change models, prompts, mode, or run count for an existing task, start another `session_dir`.


In [ ]:
experiment.status()
